# One-region parameter simulation


<details>
<summary>References</summary>

* https://github.com/the-virtual-brain/tvb-root/blob/master/tvb_library/tvb/simulator/models/zerlaut.py

* https://github.com/the-virtual-brain/tvb-root/blob/master/tvb_library/tvb/simulator/simulator.py

</details>

## Initialization



Load environment variables

<font color=#fda> NOTE: </font> Prior to loading any modules, we need to load the environment variables from the configuration files. This way, we can configure the TVB modules using environment variables.

De esta forma, podemos configurar los modulos de TVB mediante variables de entorno.

In [ ]:
from dotenv import dotenv_values, find_dotenv
import os
from pathlib import Path

defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

Modules

In [ ]:
import json5 as json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from dataclasses import asdict, dataclass #, field

In [ ]:
from tvb.basic.neotraits.api import List, NArray
from tvb.datatypes.connectivity import Connectivity
from tvb.simulator.coupling import Linear

from tvb.simulator.monitors import Raw
from tvb.simulator.simulator import Simulator

In [ ]:
#from tvb.simulator.integrators import HeunDeterministic, HeunStochastic
#from tvb.simulator.noise import Additive #, Multiplicative
#from tvb.simulator.models.zerlaut import ZerlautAdaptationFirstOrder
from tvb.simulator.models import ModelsEnum
from tvb.simulator import noise as Noise, integrators as Integrators

Console

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

Set main directory paths

In [ ]:
def resolve_directory(value, base):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (base / path).resolve()

#ToDo: Cambiar el nombe de las variables de entorno
WORK_DIR = resolve_directory(settings['TVB_WORK_DIR'], env_dir)



os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Simulation values


In [ ]:
# 1) Entradas de la simulación.

# 1.0) Salida de la simulación.
OUTPUT_DIR = resolve_directory(settings['TVB_RESULTS_DIR'], WORK_DIR)
OUTPUT_FILE = 'single_region_simulation'
OUTPUT_DTYPE = "float32"

# OUTPUT_STATES = None
# OUTPUT_STATES = ["E_e", "E_i"]
OUTPUT_STATES = ['E', 'I', 'W_e', 'W_i', 'ou_drift']

# Solo requerido para tareas como plotting.
STATE_UNITS = {
    "E": "kHz", "I": "kHz",
    "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²",
    "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
}
STIMULUS_UNITS = {
    "external_input_ex_ex": "kHz",
    "external_input_ex_in": "kHz",
    "external_input_in_ex": "kHz",
    "external_input_in_in": "kHz",
}


os.makedirs(OUTPUT_DIR, exist_ok=True)
pprint(f'Results directory: {OUTPUT_DIR}')

In [ ]:
# 1.1) Parámetros de la simulación.
DURATION_MS = 2000.0    # Duración de la simulación en milisegundos.
DT_MS = 0.5             # Paso de integración en milisegundos.

# INTEGRATOR_NAME = 'HeunStochastic'  
NOISE ={
    "name": "Additive",
    "params": {
        "nsig": [0,0,0,0,0.1]
    }
}


In [ ]:
NoiseClass = getattr(Noise, )

In [ ]:
# 1.2) Parámetros de la entrada. En este caso un pulso.
STIMULUS = {
    "external_input_ex_ex": {
        "pulse": {
            "baseline_khz": 0.005,        # Frecuencia de disparo basal
            "pulse_amplitude_khz": 0.04,  # Amplitud del pulso
            "pulse_onset_ms": 500.0,      # Instante de inicio del pulso
            "pulse_duration_ms": 800.0,   # Duración del pulso
        },
    },
    # # Define los que vas a utilizar
    # "external_input_ex_in": {
    #     "pulse_train": {
    #             "baseline_khz": 0.001,        # Frecuencia de disparo basal
    #             "pulse_amplitude_khz": 0.01,  # Amplitud del pulso
    #             "pulse_onset_ms": 500.0,      # Instante de inicio del pulso
    #             "pulse_duration_ms": 100.0,   # Duración del pulso
    #             "pulse_period_ms": 150.0,     # Periodo del tren de pulsos
    #         },
    # }, 
    
    # "external_input_in_ex": {"external_input_ex_ex": 0.5},
    # "external_input_in_in": {"external_input_ex_in": 0.5},
}


In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
#MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"


MODEL_PARAMETERS = {
    # Población y conectividad
    "N_tot": 10000,
    "p_connect_e": 0.05,
    "p_connect_i": 0.05,
    "g": 0.2,
    "K_ext_e": 400,
    "K_ext_i": 35,
    
    # Parametros de membrana
    "g_L": 10.0,
    "C_m": 200.0,
    "E_L_e": -60.0,
    "E_L_i": -65.0,

    # Parametors de adaptación
    "a_e": 4.0,
    "b_e": 60.0,
    "tau_w_e": 200.0,
    "a_i": 0.0,
    "b_i": 20.0,
    "tau_w_i": 100.0,

    # Parametros sinápticos
    "E_e": 0.0,
    "E_i": -80.0,
    "Q_e": 1.0,
    "Q_i": 5.0,
    "tau_e": 5.0,
    "tau_i": 5.0,
    
    "T": 10.0,

    # Parametro de segundo orden (no hay un motivo matemático)
    "S_i": 1, # Escala la entrada remota recibida por la población inhibidora en el modelo de segundo orden.

    # Parametros de ruido
    "weight_noise": 10.0,
    "tau_OU": 5.0,
}

# Todos los parámetros son de tipo float menos lo que están en esta lista. 
INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"}


**Parametros que se utilizaran para estimular el modelo.**

```python
[external_input_ex_ex, external_input_ex_in, 
external_input_in_ex, external_input_in_in]
```

**Parametros de ruido.**

<font color=#fda> NOTE: </font> Vamos a utilizar los valores aleatorios para generar ruido,como aumento de datos. 

```python
[tau_OU, weight_noise]
```

## Build the stimulus

In [ ]:
def _build_pulse(
        time_ms, baseline_khz, pulse_amplitude_khz, pulse_onset_ms, pulse_duration_ms
):
    stimulus = np.full(time_ms.shape, baseline_khz)
    pulse_mask = ((time_ms >= pulse_onset_ms) & 
        (time_ms < pulse_onset_ms + pulse_duration_ms))
    stimulus[pulse_mask] = pulse_amplitude_khz
    
    return stimulus

In [ ]:
def _build_pulse_train(
        time_ms, baseline_khz, pulse_amplitude_khz, pulse_onset_ms,
        pulse_duration_ms, pulse_period_ms,
):
    stimulus = np.full(time_ms.shape, baseline_khz)
    phase_ms = (time_ms - pulse_onset_ms) % pulse_period_ms
    pulse_mask = ((time_ms >= pulse_onset_ms) & 
        (phase_ms < pulse_duration_ms))
    stimulus[pulse_mask] = pulse_amplitude_khz

    return stimulus

In [ ]:
# ToDo: Esta clase devuelve arrays, pero la clase del simulador acepta un diccionario de funciones. 

def build_stimulus(stimulus, duration_ms, dt_ms):
    time_ms = np.arange(0.0, duration_ms, dt_ms)
    output = {}
    for key, value in stimulus.items():
        if value is None: 
            continue

        # ToDo: Por ahora solo se permite una señal de un tipo. 
        #       Crear tipos de estímulos que permitan combinaciones.
        if not isinstance(value, dict) or len(value) != 1:
            raise ValueError(f"Stimulus value for {key} must be a dictionary with one key.")

        match next(iter(value.keys())): # Solo hay una clave en el diccionario.
            case "pulse_train":
                output[key] = _build_pulse_train(time_ms, **value["pulse_train"])
            case "pulse":
                output[key] = _build_pulse(time_ms, **value["pulse"])
        
        # ToDo: Otros tipos de estímulos ...

    for key, value in stimulus.items():
        if value is None:
            continue

        # Ya sabemos que es un diccionario de un solo elemento.
        source, scale = next(iter(value.items())) 

        if source in stimulus:
            if source not in output:
                raise ValueError(
                    f"Stimulus value for {key} requires '{source}' to be defined."
                )

            output[key] = scale * output[source]

    return time_ms, [(key, output[key][:, np.newaxis]) for key in output]

In [ ]:
time_ms, stimulus = build_stimulus(STIMULUS, DURATION_MS, DT_MS)

In [ ]:
# Plotting

fig, ax = plt.subplots(figsize=(10, 3))

for key, values in stimulus:
    ax.plot(time_ms, values, label=key)

ax.set_title("Stimulation Signals")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Stimulus (KHz)")
ax.legend()
plt.show()

## Auxiliary functions

### New simulator class

In [ ]:
### New Simulation Class
# ToDo: Escoger nombre. DynamicParameterSimulator, ParameterStimulationSimulator,
#       TimeVaryingParameterSimulator

class DynamicParameterSimulator(Simulator):
    """Apply time varying parameter values to regions or nodes."""

    dynamic_parameters = List(
        of=tuple,
        default=(),
        label="Time varying model parameters",
        doc=(
            "Each entry is (parameter_name, source). Source is an array of "
            "shape (steps, regions), (steps, nodes), or (steps, 1), or a "
            "sequence of scalar functions for each region, each node, or all "
            "nodes. For a constant value, set the model parameter directly."
        ),
    )

    def configure(self, full_configure=True):
        # Let TVB configure the model, connectivity, and optional surface.
        super().configure(full_configure=full_configure)

        # Validate each source and remember how to read it during the run.
        n_regions, n_nodes = self.connectivity.number_of_regions, self.number_of_nodes
        self._parameter_sources, names = [], set()
    
        for name, source in self.dynamic_parameters:
            # Reject duplicate parameter names.
            if name in names:
                raise ValueError(f"Duplicate parameter: {name}")
            names.add(name)

            # Validate that the parameter exists and is a numeric array.
            descriptor = getattr(type(self.model), name, None)
            if not isinstance(descriptor, NArray):
                raise ValueError(f"Unknown numeric parameter: {name}")

            # ToDo: Validate the type of the array. 
            # ToDo:Por ahora solo funciona para floats

            # Validate the source type and shape.
            source_type = "invalid"
            if isinstance(source, np.ndarray):
                if source.ndim != 2:
                    source_type = "invalid"
                elif self.surface is not None and source.shape[1] == n_nodes: # It must be the first
                     source_type = "node_array"
                elif source.shape[1] == n_regions:
                    source_type = "region_array"
                elif source.shape[1] == 1:
                    source_type = "scalar_array" # It must be the last

            elif (
                isinstance(source, (list, tuple))
                and all(map(callable, source))
            ):
                if self.surface is not None and len(source) == n_nodes: # It must be the first
                    source_type = "node_functions"
                elif len(source) == n_regions:
                    source_type = "region_functions"
                elif len(source) == 1: # It must be the last
                    source_type = "scalar_function"
                

            if source_type == "invalid":
                raise ValueError(
                    f"{name} needs an array or one function per region or node"
                )

            self._parameter_sources.append((name, source, source_type))
        return self

    def _loop_update_stimulus(self, step, stimulus):
        super()._loop_update_stimulus(step, stimulus)

        index = step - 1  # Step 1 starts at t=0.
        time_ms = index * self.integrator.dt

        n_regions = self.connectivity.number_of_regions
        for name, source, source_type in self._parameter_sources:
            match source_type:
                case "region_array" | "node_array": 
                    values = source[index]
                case "scalar_array": 
                    values = np.full(self.number_of_nodes, source[index, 0])
                case "scalar_function": 
                    values = np.full(self.number_of_nodes, source[0](time_ms))
                case "region_functions" | "node_functions": 
                    values = [function(time_ms) for function in source]

            # ToDo: Por ahora esta restringido floats. 
            values = np.asarray(values, dtype=float)
            
            if self.surface is not None and source_type.startswith("region_"):
                values = values[self.surface.region_mapping]
   
            setattr(self.model, name, values[:, np.newaxis])

### Build simulation components

In [ ]:
def get_model_class(model_name): return ModelsEnum(model_name).get_class()
def get_integrator_class(integrator_name): return getattr(Integrators, integrator_name)
def get_noise_class(noise_name): return getattr(Noise, noise_name)

# model_name = "ZerlautAdaptationFirstOrder"

In [ ]:
def build_model(parameters, integer_parameters, 
                model_class="ZerlautAdaptationFirstOrder",
                variable_of_interest=None):
    model = get_model_class(model_class)()

    for name, value in parameters.items():
        dtype = np.int64 if name in integer_parameters else np.float64
        setattr(model, name, np.array([value], dtype=dtype))

    # Esto se usa para el monitor Raw que se utiliza para guardar la salida de la simulación.
    # Por defecto solo se guarda E.
    if variable_of_interest is not None:
        if any(var not in model.state_variables for var in variable_of_interest):
            raise ValueError(f"Some variables of interest are not state variables: {variable_of_interest}")
        
        model.variables_of_interest = variable_of_interest
    
    return model

model = build_model(MODEL_PARAMETERS | {"weight_noise": 0.0}, INTEGER_PARAMETERS)
pprint(f'Model variables of interest: {model.variables_of_interest}')
pprint(f'Model state variables: {model.state_variables}')
pprint(f'Model coupling variables: {np.asarray(model.state_variables)[model.cvar]}')
pprint(f'Model stimulus variables: {model.stvar}') # None => cvar




In [ ]:
def build_integrator(dt_ms, integrator_class = "HeunDeterministic" ,noise_config=None):
    integrator = get_integrator_class(integrator_class)(dt = dt_ms)
    
    if noise_config is not None:
        noise_class = get_noise_class(noise_config.get("name", "Additive"))
        noise = noise_class()

        if "params" in noise_config:
            for name, value in noise_config["params"].items():
                if value is None: continue
                if isinstance(value, (list, tuple)):
                    value = np.array(value, dtype=np.float64)
                setattr(noise, name, value)

        integrator.noise = noise

    return integrator

# ToDo: Parametros del integrador. Comprobar en el futuro
# integrator_class = get_integrator_class("HeunStochastic")
# for name in integrator_class.declarative_attrs:
#     pprint(name, repr(getattr(integrator_class, name)))

# noise_class = get_noise_class("Additive")
# for name in noise_class.declarative_attrs:
#     print(name, repr(getattr(noise_class, name)))

In [ ]:
def build_single_region_simulator(model, stimulus, integrator):
    return DynamicParameterSimulator(
        model=model,
        dynamic_parameters=stimulus,
        # stimulus= Value for variables in model.stvars 
        # state_variables todas, cvar acoplables, stvar estimulables.
        
        connectivity=Connectivity(
            weights=np.zeros((1, 1)),
            tract_lengths=np.zeros((1, 1)),
            speed=np.array([3.0]), # mm/ms; una velocidad común para las conexiones.
            # idelays (N, N) (steps)
            #delays` (N, N)
            centres=np.zeros((1, 3)),
            region_labels=np.array(["región"]),
        ),
        coupling=Linear(a=np.array([0.0])),
        # conduction_speed, redundante no usar

        integrator=integrator,
        monitors=(Raw(),),
        initial_conditions=np.zeros((1, 5, 1, 1)), # history, state, region, node
    ).configure()

***Listar las opciones: Modelos, integradores y ruido.***

In [ ]:
# # Posibles modelos
# pprint(f"Modelos disponibles: {sorted(model.value for model in ModelsEnum)}")


# # stoch = {n[:-10] for n in Integrators.__dict__ if n.upper().endswith('STOCHASTIC')}
# # deter = {n[:-13] for n in Integrators.__dict__ if n.upper().endswith('DETERMINISTIC')}
# # integrators = stoch | deter
# # # Seguro que todos son integradores
# # pprint(f"Integradores disponibles: {sorted(integrators)}")
# # No todos son ingradores
# # pprint(f"Integradores disponibles: {[n for n in Integrators.__dict__] }")

# integrator_names = sorted(
#     name for name, cls in vars(Integrators).items()
#     if isinstance(cls, type)
#     and issubclass(cls, Integrators.Integrator)
# )
# pprint(f"Integradores disponibles: {integrator_names}") # HeunDeterministic, HeunStochastic


# noise_names = [
#     name for name, cls in vars(Noise).items()
#     if isinstance(cls, type)
#     and issubclass(cls, Noise.Noise)
# ]
# pprint(f"Ruido disponible: {noise_names}") # Additive, Multiplicative


### Save and load resuls


In [ ]:
@dataclass
class SimulationMetadata:
    duration_ms: float
    dt_ms: float
    integrator: str = "HeunDeterministic"
    output_dtype: str = "float32"
    noise_type: str = "Additive"
    noise_params: dict = None
    
    def __post_init__(self):
        if not isinstance(self.integrator, str):
            self.integrator = type(self.integrator).__name__

@dataclass
class ModelMetadata:
    name: str = "ZerlautAdaptationFirstOrder"
    parameters: dict[str, float | int] = None
    integer_parameters: list[str] = None
    output_variables: list[str] = None
    state_units: dict[str, str] = None
    stimulus_units: dict[str, str] = None
        
    def __post_init__(self):
        if not isinstance(self.name, str):
            self.name = type(self.name).__name__

@dataclass
class ResultMetadata:
    simulation: SimulationMetadata
    model: ModelMetadata
    stimulus: dict


In [ ]:
def save_single_region_simulation(
    file_path, time_ms, simulation_output, metadata,
):
    output_variables = metadata.model.output_variables

    # ToDo: Faltan muchas comprobaciones.
    if len(output_variables) != simulation_output.shape[1]:
        raise ValueError("Output variables do not match simulation output shape")

    result = {"time_ms": time_ms}
    result.update(
        {
            var: np.asarray(simulation_output[:, i], dtype=metadata.simulation.output_dtype) 
            for i, var in enumerate(output_variables)
        }
    )

    (file_path := Path(file_path)).parent.mkdir(parents=True, exist_ok=True)
    
    np.savez(
        file_path, **result,
        metadata_json=np.array(json.dumps(asdict(metadata))),
    )

In [ ]:
def load_results(path):
    with np.load(path, allow_pickle=False) as saved:
        data = json.loads(saved["metadata_json"].item())
        metadata = ResultMetadata(
            simulation=SimulationMetadata(**data["simulation"]),
            model=ModelMetadata(**data["model"]),
            stimulus=data["stimulus"],
        )
        times = np.asarray(saved["time_ms"], dtype=float)
        states = {
            name: np.asarray(saved[name], dtype=float)
            for name in metadata.model.output_variables
        }

    return {"time_ms": times, "states": states, "metadata": metadata}


### Plot results

In [ ]:
def plot_results(times, states, metadata, show_stimulus=True, ax=None, plot_states=None):
    times = np.asarray(times)
    if plot_states is None:
        plotted_states = states
    else:
        if isinstance(plot_states, str):
            plot_states = [plot_states]
        plotted_states = {name: states[name] for name in plot_states}

    state_units = metadata.model.state_units or {}
    stimulus_units = metadata.model.stimulus_units or {}
    groups = []

    if show_stimulus:
        stimulus_times, stimulus_data = build_stimulus(
            metadata.stimulus,
            metadata.simulation.duration_ms,
            metadata.simulation.dt_ms,
        )
        units = dict.fromkeys(stimulus_units.get(name, "model units")
                              for name, _ in stimulus_data)
        for unit in units:
            signals = [(name, values) for name, values in stimulus_data
                       if stimulus_units.get(name, "model units") == unit]
            groups.append(("Stimuli", unit, signals))

    units = dict.fromkeys(state_units.get(name, "model units") for name in plotted_states)
    for unit in units:
        names = [name for name in plotted_states
                 if state_units.get(name, "model units") == unit]
        groups.append(("State variables", unit, names))

    if ax is None:
        fig, axes = plt.subplots(
            len(groups), 1, figsize=(11, 3 * len(groups)),
            sharex=True, squeeze=False, constrained_layout=True,
        )
        axes = axes[:, 0]
    else:
        fig = ax.figure
        subplotspec = ax.get_subplotspec()
        position = ax.get_position()
        ax.remove()
        if subplotspec is None:
            grid = fig.add_gridspec(
                len(groups), 1, left=position.x0, right=position.x1,
                bottom=position.y0, top=position.y1,
            )
        else:
            grid = subplotspec.subgridspec(len(groups), 1)
        axes = []
        for i in range(len(groups)):
            axes.append(fig.add_subplot(grid[i], sharex=axes[0] if axes else None))
        axes = np.asarray(axes)

    for axis, (kind, unit, data) in zip(axes, groups):
        if kind == "Stimuli":
            edges = np.r_[stimulus_times, times[-1]]
            for name, values in data:
                axis.stairs(np.asarray(values).ravel(), edges, label=name)
        else:
            for name in data:
                axis.plot(times, plotted_states[name], label=name)
        axis.set_ylabel(f"{kind} ({unit})")
        axis.grid(alpha=0.25)
        axis.legend()
    axes[-1].set_xlabel("Time (ms)")
    return fig, axes

## Deterministic Simulation

### Run simulaiton

In [ ]:
# integrator = HeunStochastic(dt=0.1, noise=Additive(nsig=np.array([0,0,0,0,0.01])))
# integrator = HeunDeterministic(dt=0.1)

integrator = build_integrator(DT_MS, integrator_class="HeunDeterministic")# , noise_config=NOISE)

In [ ]:

model = build_model(
    MODEL_PARAMETERS, INTEGER_PARAMETERS, 
    model_class="ZerlautAdaptationFirstOrder",
    variable_of_interest=OUTPUT_STATES
)

simulator = build_single_region_simulator(model, stimulus, integrator)

pprint('Configured sources:', [(name, kind) for name, _, kind in simulator._parameter_sources])

In [ ]:
time_detsim_raw, detsim_raw = simulator.run(n_steps=len(time_ms))[0] 
detsim_output = detsim_raw[:, :, 0, 0]  # history, state, region, node,

pprint(f"Simulation output shape: {detsim_output.shape}")

In [ ]:
pprint(f"Simulation steps: {time_detsim_raw.shape}")
pprint(f"Stimulation steps: {time_ms.shape}")

#Ojo parece que hay un desfase y esto puede ser un problema.
pprint(f"Maximum difference: {np.max(np.abs(time_detsim_raw - time_ms))}")
pprint(f"Minimum difference: {np.min(np.abs(time_detsim_raw - time_ms))}")

### Save results

In [ ]:
output_variables = list(model.variables_of_interest if OUTPUT_STATES is None else OUTPUT_STATES)
selected_output = detsim_output[:, [model.state_variables.index(name) for name in output_variables]]

metadata = ResultMetadata(
    simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
    model=ModelMetadata(
        name=model,
        parameters=MODEL_PARAMETERS,
        integer_parameters=list(INTEGER_PARAMETERS), # No puedes descargar sets
        output_variables=output_variables,
        state_units=STATE_UNITS,
        stimulus_units=STIMULUS_UNITS,
    ),
    stimulus=STIMULUS,
)

OUTPUT_PATH = OUTPUT_DIR / f"{OUTPUT_FILE}_det1.npz"
save_single_region_simulation(
    OUTPUT_PATH, time_detsim_raw, selected_output, metadata,
)

pprint(f"Saved: '{OUTPUT_PATH}'")


### Load results

In [ ]:
loaded_results = load_results(OUTPUT_PATH)
pprint(f"Loaded {len(loaded_results['time_ms'])} samples from '{OUTPUT_PATH}'")
pprint("State variables:", list(loaded_results["states"]))
pprint("Metadata:", type(loaded_results["metadata"]).__name__)


### Plot results


In [ ]:
PLOT_STATES = None  # Por ejemplo: ["E", "W_e"]

fig, axes = plot_results(
    loaded_results["time_ms"],
    loaded_results["states"],
    loaded_results["metadata"],
    show_stimulus=True,
    plot_states=PLOT_STATES,
)
plt.show()


## Stochastic Simulation

### Run simulaiton

In [ ]:
# integrator = HeunStochastic(dt=0.1, noise=Additive(nsig=np.array([0,0,0,0,0.01])))
# integrator = HeunDeterministic(dt=0.1)


NOISE |= {"params": {"nsig": [0, 0, 0, 0, 0.0001]}} 
integrator = build_integrator(DT_MS, integrator_class="HeunStochastic", noise_config=NOISE)

In [ ]:

model = build_model(
    MODEL_PARAMETERS, INTEGER_PARAMETERS, 
    model_class="ZerlautAdaptationFirstOrder",
    variable_of_interest=OUTPUT_STATES
)

simulator = build_single_region_simulator(model, stimulus, integrator)

pprint('Configured sources:', [(name, kind) for name, _, kind in simulator._parameter_sources])

In [ ]:
time_detsim_raw, detsim_raw = simulator.run(n_steps=len(time_ms))[0] 
detsim_output = detsim_raw[:, :, 0, 0]  # history, state, region, node,

pprint(f"Simulation output shape: {detsim_output.shape}")

In [ ]:
pprint(f"Simulation steps: {time_detsim_raw.shape}")
pprint(f"Stimulation steps: {time_ms.shape}")

#Ojo parece que hay un desfase y esto puede ser un problema.
pprint(f"Maximum difference: {np.max(np.abs(time_detsim_raw - time_ms))}")
pprint(f"Minimum difference: {np.min(np.abs(time_detsim_raw - time_ms))}")

### Save results

In [ ]:
output_variables = list(model.variables_of_interest if OUTPUT_STATES is None else OUTPUT_STATES)
selected_output = detsim_output[:, [model.state_variables.index(name) for name in output_variables]]

metadata = ResultMetadata(
    simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
    model=ModelMetadata(
        name=model,
        parameters=MODEL_PARAMETERS,
        integer_parameters=list(INTEGER_PARAMETERS), # No puedes descargar sets
        output_variables=output_variables,
        state_units=STATE_UNITS,
        stimulus_units=STIMULUS_UNITS,
    ),
    stimulus=STIMULUS,
)

OUTPUT_PATH = OUTPUT_DIR / f"{OUTPUT_FILE}_stoch1.npz"
save_single_region_simulation(
    OUTPUT_PATH, time_detsim_raw, selected_output, metadata,
)

pprint(f"Saved: '{OUTPUT_PATH}'")


### Load results

In [ ]:
loaded_results = load_results(OUTPUT_PATH)
pprint(f"Loaded {len(loaded_results['time_ms'])} samples from '{OUTPUT_PATH}'")
pprint("State variables:", list(loaded_results["states"]))
pprint("Metadata:", type(loaded_results["metadata"]).__name__)


### Plot results


In [ ]:
PLOT_STATES = None  # Por ejemplo: ["E", "W_e"]

fig, axes = plot_results(
    loaded_results["time_ms"],
    loaded_results["states"],
    loaded_results["metadata"],
    show_stimulus=True,
    plot_states=PLOT_STATES,
)
plt.show()
